# Phase 1: Species Synonyms

This notebook shows the Phase 1 pipeline stages directly: load settings, acquire LPSN records, normalize them, and write release tables. The reusable work stays in `pathogen_atlas_v2`; the notebook makes the stage order and the functions being called visible.

Set `run_phase: true` in `configs/notebook_parameters.yaml` to run acquisition. The default is `false`.

In [9]:
from dataclasses import replace
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2 import notebook_utils as nbutils
from pathogen_atlas_v2.abstracts.species_queries import build_species_pubmed_query
from pathogen_atlas_v2.common.io import resolve_path
from pathogen_atlas_v2.species.acquire import (
    SpeciesAcquisitionError,
    acquire_lpsn_species_records,
    load_species_source_config,
    utc_now_iso,
)
from pathogen_atlas_v2.species.export_mappings import write_release_outputs
from pathogen_atlas_v2.species.normalise import normalise_lpsn_records

## Settings and release paths

The source config defines LPSN credentials, raw caches, Phase 1 release paths, and optional V1 comparison inputs. `run_phase1` in the package uses the same stage functions shown below; these cells expose their order and outputs for inspection.

In [10]:
NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase1_species_synonyms", NOTEBOOK_CONFIG_PATH)

CONFIG_PATH = PARAMS["config_path"]
FORCE_REFRESH = bool(PARAMS["force_refresh"])
RESUME = bool(PARAMS["resume"])
RUN_PHASE = bool(PARAMS["run_phase"])
ENRICH_TAXONOMY_HIERARCHY = PARAMS.get("enrich_taxonomy_hierarchy")

phase1_config = load_species_source_config(CONFIG_PATH)
if ENRICH_TAXONOMY_HIERARCHY is not None:
    phase1_config = replace(
        phase1_config,
        enrich_taxonomy_hierarchy=bool(ENRICH_TAXONOMY_HIERARCHY),
    )
phase1_paths = nbutils.phase1_output_paths(CONFIG_PATH)

display(pd.DataFrame([{
    "source": phase1_config.name,
    "category": phase1_config.category,
    "run_phase": RUN_PHASE,
    "force_refresh": FORCE_REFRESH,
    "resume": RESUME,
    "enrich_taxonomy_hierarchy": phase1_config.enrich_taxonomy_hierarchy,
    "credentials_path": str(phase1_config.credentials_path),
    "raw_records_path": str(phase1_config.raw_records_path),
}]))
display(nbutils.existing_outputs(phase1_paths))

,source,category,run_phase,force_refresh,resume,enrich_taxonomy_hierarchy,credentials_path,raw_records_path
0,lpsn,species,False,False,True,True,/Users/f.costa/Documents/Projects/Repositories...,/Users/f.costa/Documents/Projects/Repositories...


,name,path,exists,rows,size_bytes,content_type
0,candidate_records,/Users/f.costa/Documents/Projects/Repositories...,True,28782,41767188,jsonl
1,raw_records,/Users/f.costa/Documents/Projects/Repositories...,True,28782,516189722,jsonl
2,lookup_cache,/Users/f.costa/Documents/Projects/Repositories...,True,13969,19897584,jsonl
3,progress,/Users/f.costa/Documents/Projects/Repositories...,True,1,690,json
4,normalised_records,/Users/f.costa/Documents/Projects/Repositories...,True,28782,1486892,parquet
5,species_canonical,/Users/f.costa/Documents/Projects/Repositories...,True,22804,640900,parquet
6,species_synonym_mapping,/Users/f.costa/Documents/Projects/Repositories...,True,28766,1029834,parquet
7,species_mapping_conflicts,/Users/f.costa/Documents/Projects/Repositories...,True,16,7144,parquet
8,species_taxonomy,/Users/f.costa/Documents/Projects/Repositories...,True,22804,402070,parquet
9,metadata,/Users/f.costa/Documents/Projects/Repositories...,True,1,3005,json


## Stage 1: Acquire or resume LPSN records

This calls the source client and its checkpoint/resume logic. If the run flag is off, no acquisition starts.

In [11]:
phase1_retrieved_at = None
phase1_raw_records = None
phase1_metadata = {"status": "skipped", "reason": "Set run_phase: true to run Phase 1."}

if RUN_PHASE:
    phase1_retrieved_at = utc_now_iso()
    try:
        phase1_raw_records = acquire_lpsn_species_records(
            config=phase1_config,
            force_refresh=FORCE_REFRESH,
            resume=RESUME,
            log=True,
        )
        print(f"Acquired or resumed {len(phase1_raw_records):,} LPSN records.")
    except SpeciesAcquisitionError as exc:
        phase1_metadata = {"status": "not_run", "reason": str(exc)}
        print(f"Phase 1 acquisition could not run: {exc}")
else:
    print(phase1_metadata["reason"])

display(nbutils.phase1_progress_status(CONFIG_PATH))

Set run_phase: true to run Phase 1.


,status,mode,retrieved_records,completed_records,total_records,remaining_records,percent_complete,enrich_taxonomy_hierarchy,updated_at,path
0,completed,enrichment_backfill,28782,28782,28782,0,100.0,True,2026-06-27T13:00:05+00:00,/Users/f.costa/Documents/Projects/Repositories...


## Stage 2: Normalize taxonomic records

Normalization maps each input name to the accepted canonical name and extracts stable taxonomy fields. The table preview makes the result inspectable before release export.

In [12]:
phase1_normalised = None
if phase1_raw_records is not None:
    phase1_normalised = normalise_lpsn_records(
        phase1_raw_records,
        source=phase1_config.name,
        retrieved_at=phase1_retrieved_at,
    )
    print(f"Normalized to {len(phase1_normalised):,} rows.")
    display(phase1_normalised.head(20))
else:
    print("Normalization skipped because no raw records were acquired in this notebook run.")

Normalization skipped because no raw records were acquired in this notebook run.


## Stage 3: Build and write release tables

The export function builds canonical-species, synonym, taxonomy, conflict, and metadata outputs. It also reports how the new mapping compares with configured V1 reference files.

In [13]:
if phase1_normalised is not None:
    phase1_metadata = write_release_outputs(
        config=phase1_config,
        normalised_df=phase1_normalised,
        raw_record_count=len(phase1_raw_records),
        retrieved_at=phase1_retrieved_at,
    )
    print("Phase 1 release outputs written.")

phase1_summary = nbutils.phase_run_summary(phase1_metadata)
display(phase1_summary["overview"])
display(phase1_summary["counts"])
display(nbutils.metadata_package_versions(phase1_metadata))
display(nbutils.metadata_outputs(phase1_metadata))
display(nbutils.metadata_v1_comparison(phase1_metadata))
display(nbutils.existing_outputs(phase1_paths))

,field,value
0,status,skipped
1,reason,Set run_phase: true to run Phase 1.


,status
0,no_counts


,status
0,no_package_versions


,status
0,no_outputs


,status
0,no_v1_comparison


,name,path,exists,rows,size_bytes,content_type
0,candidate_records,/Users/f.costa/Documents/Projects/Repositories...,True,28782,41767188,jsonl
1,raw_records,/Users/f.costa/Documents/Projects/Repositories...,True,28782,516189722,jsonl
2,lookup_cache,/Users/f.costa/Documents/Projects/Repositories...,True,13969,19897584,jsonl
3,progress,/Users/f.costa/Documents/Projects/Repositories...,True,1,690,json
4,normalised_records,/Users/f.costa/Documents/Projects/Repositories...,True,28782,1486892,parquet
5,species_canonical,/Users/f.costa/Documents/Projects/Repositories...,True,22804,640900,parquet
6,species_synonym_mapping,/Users/f.costa/Documents/Projects/Repositories...,True,28766,1029834,parquet
7,species_mapping_conflicts,/Users/f.costa/Documents/Projects/Repositories...,True,16,7144,parquet
8,species_taxonomy,/Users/f.costa/Documents/Projects/Repositories...,True,22804,402070,parquet
9,metadata,/Users/f.costa/Documents/Projects/Repositories...,True,1,3005,json


## Inspect release tables

In [14]:
display(nbutils.read_table_if_exists(phase1_paths["species_canonical"]))
display(nbutils.read_table_if_exists(phase1_paths["species_synonym_mapping"]))
display(nbutils.read_table_if_exists(phase1_paths["species_taxonomy"]))

,canonical_species,genus,specific_epithet,subspecies_epithet,lpsn_record_id,status,risk_group,source,source_retrieved_at
0,Abditibacterium utsteinense,Abditibacterium,utsteinense,,797965,correct name,None,lpsn,2026-06-27T12:58:04+00:00
1,Abiotrophia defectiva,Abiotrophia,defectiva,,772468,correct name (and explicitly recommended for m...,None,lpsn,2026-06-27T12:58:04+00:00
2,Absicoccus intestinalis,Absicoccus,intestinalis,,69450,correct name,None,lpsn,2026-06-27T12:58:04+00:00
3,Absicoccus porci,Absicoccus,porci,,5758,correct name,None,lpsn,2026-06-27T12:58:04+00:00
4,Abyssalbus ytuae,Abyssalbus,ytuae,,29444,correct name,None,lpsn,2026-06-27T12:58:04+00:00
5,Abyssibacter profundi,Abyssibacter,profundi,,797925,correct name,None,lpsn,2026-06-27T12:58:04+00:00
6,Abyssibius alkaniclasticus,Abyssibius,alkaniclasticus,,36886,correct name,None,lpsn,2026-06-27T12:58:04+00:00
7,Abyssicoccus albus,Abyssicoccus,albus,,794216,correct name,None,lpsn,2026-06-27T12:58:04+00:00
8,Abyssisolibacter fermentans,Abyssisolibacter,fermentans,,55696,correct name,None,lpsn,2026-06-27T12:58:04+00:00
9,Acanthopleuribacter pedis,Acanthopleuribacter,pedis,,787601,correct name,None,lpsn,2026-06-27T12:58:04+00:00


,input_name,canonical_species,mapping_type,input_lpsn_record_id,canonical_lpsn_record_id,status,source
0,Abditibacterium utsteinense,Abditibacterium utsteinense,accepted,797965,797965,correct name,lpsn
1,Abiotrophia adiacens,Granulicatella adiacens,synonym,772466,776611,synonym (and not recommended for medical use),lpsn
2,Abiotrophia balaenopterae,Granulicatella balaenopterae,synonym,772467,776612,synonym (and not recommended for medical use),lpsn
3,Abiotrophia defectiva,Abiotrophia defectiva,accepted,772468,772468,correct name (and explicitly recommended for m...,lpsn
4,Abiotrophia elegans,Granulicatella elegans,synonym,772469,776613,synonym (and not recommended for medical use),lpsn
5,Absicoccus intestinalis,Absicoccus intestinalis,accepted,69450,69450,correct name,lpsn
6,Absicoccus porci,Absicoccus porci,accepted,5758,5758,correct name,lpsn
7,Absiella argi,Amedibacterium intestinale,synonym,15254,8291,synonym,lpsn
8,Abyssalbus ytuae,Abyssalbus ytuae,accepted,29444,29444,correct name,lpsn
9,Abyssibacter profundi,Abyssibacter profundi,accepted,797925,797925,correct name,lpsn


,canonical_species,domain,kingdom,phylum,class,order,family,genus,source
0,Abditibacterium utsteinense,Bacteria,Pseudomonadati,Abditibacteriota,Abditibacteriia,Abditibacteriales,Abditibacteriaceae,Abditibacterium,lpsn
1,Abiotrophia defectiva,Bacteria,Bacillati,Bacillota,Bacilli,Lactobacillales,Aerococcaceae,Abiotrophia,lpsn
2,Absicoccus intestinalis,Bacteria,Bacillati,Bacillota,Erysipelotrichia,Erysipelotrichales,Erysipelotrichaceae,Absicoccus,lpsn
3,Absicoccus porci,Bacteria,Bacillati,Bacillota,Erysipelotrichia,Erysipelotrichales,Erysipelotrichaceae,Absicoccus,lpsn
4,Abyssalbus ytuae,Bacteria,Pseudomonadati,Bacteroidota,Flavobacteriia,Flavobacteriales,Flavobacteriaceae,Abyssalbus,lpsn
5,Abyssibacter profundi,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Nevskiales,Salinisphaeraceae,Abyssibacter,lpsn
6,Abyssibius alkaniclasticus,Bacteria,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Rhodobacterales,Paracoccaceae,Abyssibius,lpsn
7,Abyssicoccus albus,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Abyssicoccus,lpsn
8,Abyssisolibacter fermentans,Bacteria,Bacillati,Bacillota,Clostridia,Eubacteriales,Clostridiaceae,Abyssisolibacter,lpsn
9,Acanthopleuribacter pedis,Bacteria,Pseudomonadati,Acidobacteriota,Holophagae,Acanthopleuribacterales,Acanthopleuribacteraceae,Acanthopleuribacter,lpsn


## Expand a bacterial PubMed query with known synonyms

Enter a canonical species name or known synonym. The helper resolves the canonical name and produces an exact-phrase OR clause over all known names.

In [15]:
SPECIES_MAPPING_ASSET = PROJECT_ROOT / "assets/species/species_synonym_mapping.parquet"
if not SPECIES_MAPPING_ASSET.exists():
    SPECIES_MAPPING_ASSET = phase1_paths["species_synonym_mapping"]

if SPECIES_MAPPING_ASSET.exists():
    species_mapping = pd.read_parquet(SPECIES_MAPPING_ASSET)
    BACTERIAL_NAME = "Abiotrophia adiacens"  # Replace with a canonical name or known synonym.
    expanded_bacterial_query = build_species_pubmed_query(BACTERIAL_NAME, species_mapping)
    display(pd.DataFrame([{
        "input_name": expanded_bacterial_query.input_name,
        "canonical_species": expanded_bacterial_query.canonical_species,
        "known_names": "; ".join(expanded_bacterial_query.known_names),
        "field": expanded_bacterial_query.field,
        "query": expanded_bacterial_query.query,
    }]))
else:
    print(f"No synonym mapping is available yet: {SPECIES_MAPPING_ASSET}")

,input_name,canonical_species,known_names,field,query
0,Abiotrophia adiacens,Granulicatella adiacens,Abiotrophia adiacens; Granulicatella adiacens;...,Title/Abstract,"(""Abiotrophia adiacens""[Title/Abstract] OR ""Gr..."
